In [ ]:
# %% [markdown]
# # SBA Analytics Engineering & Credit Risk Segmentation
# **Author:** Raymond Sibanda (FRM)  
# **Objective:** Process 30 years of SBA 7(a) and 504 data, and apply K-Means clustering to uncover portfolio concentration risks.
# 
# ## 1. Setup & Data Loading
# We load only the necessary columns to prevent memory crashes (Out of Memory errors) when processing millions of records.

# %%

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import os
import gc
import pyarrow.parquet as pq

BASE_PATH = r"C:\CreditRiskCourse\Project_8_SBA_Analytics_Engineering\data\processed"
REPORTS_PATH = r"C:\CreditRiskCourse\Project_8_SBA_Analytics_Engineering\reports"
os.makedirs(REPORTS_PATH, exist_ok=True)

def run_portfolio_analysis(data_path, prefix, title_name):
    print(f"\n{'='*60}\nRUNNING ANALYSIS: {title_name}\n{'='*60}")
    if not os.path.exists(data_path):
        print(f"❌ File not found: {data_path}")
        return

    # 1. Load Data Safely
    cols_needed = ['LoanStatus', 'NaicsCode', 'GrossApproval', 'TermInMonths', 'JobsSupported', 'InitialInterestRate', 'BorrState', 'ProjectState']
    schema = pq.read_schema(data_path)
    available_cols = [c for c in cols_needed if c in schema.names]
    state_col = 'BorrState' if 'BorrState' in schema.names else 'ProjectState'
    if state_col not in available_cols: available_cols.append(state_col)
    
    df = pd.read_parquet(data_path, columns=available_cols)
    
    # 2. Clean Target
    df['Is_Default'] = df['LoanStatus'].apply(lambda x: 1 if str(x).strip() in ['CHGOFF', 'CHGOFF.'] else 0)
    df = df[df['LoanStatus'].isin(['CHGOFF', 'P I F', 'CHGOFF.', 'PIF'])].copy()

    # ============================================================
    # PART 1: INDUSTRY CONCENTRATION (NAICS)
    # ============================================================
    print("Generating NAICS Sector Risk Chart...")
    df['NAICS_Sector'] = df['NaicsCode'].astype(str).str[:2]
    naics_map = {
        '11': 'Agriculture', '21': 'Mining', '22': 'Utilities', '23': 'Construction',
        '31': 'Manufacturing', '32': 'Manufacturing', '33': 'Manufacturing',
        '42': 'Wholesale Trade', '44': 'Retail Trade', '45': 'Retail Trade',
        '48': 'Transportation', '49': 'Warehousing', '51': 'Information',
        '52': 'Finance', '53': 'Real Estate', '54': 'Professional Svc',
        '55': 'Management', '56': 'Admin & Support', '61': 'Education',
        '62': 'Health Care', '71': 'Arts & Entertainment', '72': 'Food & Accommodation',
        '81': 'Other Services', '92': 'Public Admin'
    }
    df['NAICS_Name'] = df['NAICS_Sector'].map(naics_map).fillna('Other')

    industry_summary = df.groupby('NAICS_Name').agg(
        Total_Loans=('Is_Default', 'count'),
        Default_Rate=('Is_Default', 'mean')
    ).reset_index()
    industry_summary = industry_summary[industry_summary['Total_Loans'] >= 100].sort_values(by='Default_Rate', ascending=True)

    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(industry_summary['NAICS_Name'], industry_summary['Default_Rate'], color='#e74c3c', edgecolor='black')
    ax.set_title(f'{title_name}: Default Rate by Industry Sector', fontsize=16, fontweight='bold', pad=20)
    ax.set_xlabel('Default Rate (%)', fontsize=12, fontweight='bold')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0%}'.format(x)))
    for bar in bars:
        width = bar.get_width()
        ax.text(width + 0.002, bar.get_y() + bar.get_height()/2, f'{width:.1%}', va='center', fontweight='bold', fontsize=11)
    ax.set_xlim(0, industry_summary['Default_Rate'].max() * 1.15)
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_PATH, f"{prefix}_1_NAICS_Sector_Risk.png"), dpi=300)
    plt.close()

    # ============================================================
    # PART 2: K-MEANS CLUSTERING
    # ============================================================
    print("Running K-Means Clustering...")
    cluster_cols = ['GrossApproval', 'TermInMonths', 'JobsSupported', 'InitialInterestRate']
    cluster_cols = [c for c in cluster_cols if c in df.columns]
    df_cluster = df[cluster_cols].dropna().copy()

    for col in ['GrossApproval', 'JobsSupported']:
        if col in df_cluster.columns:
            df_cluster[col] = df_cluster[col].clip(upper=df_cluster[col].quantile(0.99))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(df_cluster)
    
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X_scaled)
    df_pca = pd.DataFrame(X_pca, columns=['PCA1', 'PCA2'])

    fig, axes = plt.subplots(1, 3, figsize=(20, 6))
    fig.suptitle(f'{title_name}: Portfolio Segmentation (K=4, 5, 6)', fontsize=18, fontweight='bold')
    K_values = [4, 5, 6]
    colors = ['viridis', 'magma', 'coolwarm']
    x_lim = (np.percentile(df_pca['PCA1'], 1), np.percentile(df_pca['PCA1'], 99))
    y_lim = (np.percentile(df_pca['PCA2'], 1), np.percentile(df_pca['PCA2'], 99))

    for i, k in enumerate(K_values):
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = kmeans.fit_predict(X_scaled)
        df_pca['Cluster'] = labels
        sns.scatterplot(data=df_pca, x='PCA1', y='PCA2', hue='Cluster', palette=colors[i], alpha=0.4, s=5, ax=axes[i], legend='brief')
        axes[i].set_title(f'K = {k} Clusters', fontsize=14)
        axes[i].set_xlim(x_lim)
        axes[i].set_ylim(y_lim)
        sns.move_legend(axes[i], "upper left", bbox_to_anchor=(1, 1))
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_PATH, f"{prefix}_2_KMeans_Comparison.png"), dpi=300)
    plt.close()

    # Profile K=4
    kmeans_4 = KMeans(n_clusters=4, random_state=42, n_init=10)
    df_cluster['Cluster'] = kmeans_4.fit_predict(X_scaled)
    profile = df_cluster.groupby('Cluster')[cluster_cols].mean().round(2)
    profile['Count'] = df_cluster.groupby('Cluster').size()
    print(f"\n{title_name} - K=4 Cluster Profiles:")
    print(profile.sort_values(by='GrossApproval', ascending=False).to_string())

    # ============================================================
    # PART 3: GEOGRAPHIC CONCENTRATION
    # ============================================================
    print("Generating State Concentration...")
    state_summary = df.groupby(state_col).agg(
        Total_Loans=('Is_Default', 'count'),
        Total_Exposure_B=('GrossApproval', lambda x: x.sum() / 1e9),
        Default_Rate=('Is_Default', 'mean')
    ).reset_index()
    state_summary = state_summary.sort_values(by='Total_Exposure_B', ascending=False).head(15)

    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    ax1.bar(state_summary[state_col], state_summary['Total_Exposure_B'], color='#3498db', alpha=0.7)
    ax1.set_ylabel('Total Exposure (Billions $)', color='#3498db', fontsize=12, fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='#3498db')
    ax1.set_xlabel('Borrower State', fontsize=12, fontweight='bold')
    
    ax2 = ax1.twinx()
    ax2.plot(state_summary[state_col], state_summary['Default_Rate'], color='#e74c3c', marker='o', linewidth=2)
    ax2.set_ylabel('Default Rate (%)', color='#e74c3c', fontsize=12, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor='#e74c3c')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: '{:.0%}'.format(x)))
    
    plt.title(f'{title_name}: Top 15 States by Exposure & Default Risk', fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(os.path.join(REPORTS_PATH, f"{prefix}_3_State_Concentration.png"), dpi=300)
    plt.close()

    del df, df_cluster, df_pca
    gc.collect()

# ============================================================
# EXECUTE FOR BOTH PORTFOLIOS
# ============================================================
run_portfolio_analysis(
    data_path=os.path.join(BASE_PATH, "sba_7a_consolidated.parquet"), 
    prefix="7a", 
    title_name="SBA 7(a) Working Capital"
)

run_portfolio_analysis(
    data_path=os.path.join(BASE_PATH, "sba_504_consolidated.parquet"), 
    prefix="504", 
    title_name="SBA 504 Asset/Real Estate"
)


RUNNING ANALYSIS: SBA 7(a) Working Capital
Generating NAICS Sector Risk Chart...
Running K-Means Clustering...

SBA 7(a) Working Capital - K=4 Cluster Profiles:
         GrossApproval  TermInMonths  JobsSupported  InitialInterestRate   Count
Cluster                                                                         
1           1232356.72        270.90          10.22                 5.73   76854
2            907744.69        117.00          65.36                 5.93   28250
3            192556.30         87.28           7.38                 5.85  295864
0             75879.01         86.63           4.49                 9.03  135159
Generating State Concentration...

RUNNING ANALYSIS: SBA 504 Asset/Real Estate
Generating NAICS Sector Risk Chart...
Running K-Means Clustering...

SBA 504 Asset/Real Estate - K=4 Cluster Profiles:
         GrossApproval  TermInMonths  JobsSupported  Count
Cluster                                                   
3           1420886.01        244.27